# Workflow 1: Identify Native Non-Covalent Lasso Entanglements (NCLEs)

## Goal

Identify all native NCLEs present in a reference protein structure, filter for high-confidence entanglements, cluster redundant variants, and compute structural features for the representative set.

---

## Typical runtime

| Step | Runtime |
|------|---------|
| Native entanglement detection | 1–10 minutes |
| High-quality filtering | < 1 minute |
| Clustering | < 1 minute |
| Feature generation | 1–5 minutes |

---

## Required input files

| File | Path | Role |
|------|------|------|
| Cleaned all-atom PDB | `$DATASTORE/user_input/reference_structures/1zmr_model_clean.pdb` | Input structure |
| Cα COR coordinates | `$DATASTORE/user_input/reference_structures/1zmr_model_clean_ca.cor` | Coordinate reference |

> **Note:** This tutorial demonstrates analysis on an all-atom structure, but NCLEdetector can also handle Cα coarse-grained structures.

---

> **Environment:** Before running this notebook, activate the `ncledetector` conda environment and register it as a kernel:
```bash
conda activate ncledetector
python -m ipykernel install --user --name ncledetector --display-name "ncledetector"
```
> Then select **ncledetector** from the kernel picker (top-right of VS Code or Jupyter).

## Step 0. Activate your environment and set paths

Set the DATASTORE root and create all output directories needed for this workflow.  
*(The `conda activate` command is handled outside the notebook — see the environment note above.)*

In [1]:
import os

DATASTORE = "/home/ians/STORE/NCLEdetector"
OUTDIR = f"{DATASTORE}/outputs/notebook_tmp/workflow1"
NCLE_outdir = f"{OUTDIR}/NCLE"
HQ_NCLE_outdir = f"{OUTDIR}/HQ_NCLE"
clustered_HQ_NCLE_outdir = f"{OUTDIR}/clustered_HQ_NCLE"
clustered_HQ_NCLE_feature_outdir = f"{OUTDIR}/clustered_HQ_NCLE_features"

os.makedirs(NCLE_outdir,                        exist_ok=True)
os.makedirs(HQ_NCLE_outdir,                    exist_ok=True)
os.makedirs(clustered_HQ_NCLE_outdir,          exist_ok=True)
os.makedirs(clustered_HQ_NCLE_feature_outdir, exist_ok=True)

print(f"DATASTORE : {DATASTORE}")
print(f"OUTDIR    : {OUTDIR}")
print("Output directories ready.")

DATASTORE : /home/ians/STORE/NCLEdetector
OUTDIR    : /home/ians/STORE/NCLEdetector/outputs/notebook_tmp/workflow1
Output directories ready.


## Step 1. Prepare a cleaned structure

The `1zmr_model_clean.pdb` file in `$DATASTORE/user_input/reference_structures/` is already cleaned and ready to use. For your own protein you would:

- rebuild missing residues and atoms (e.g., with **Modeller** or **CHARMM**);
- remove duplicate residue records;
- verify chain labels and residue numbering are sensible;
- ensure the PDB contains only the protein chain(s) of interest.

## Step 2. Detect native non-covalent lasso entanglements (NCLEs)

Scans the input structure and identifies all **native non-covalent lasso entanglements** present in the reference conformation. Each NCLE is characterized by a loop closed by a disulfide or non-covalent contact and one or more chain termini that thread through it.

**Practical notes:**
- If your structure is all-atom (conventional PDB), keep `Calpha=False` and `CG=False`.
- If your structure is all-atom and you wish to use the alpha carbons to define contacts rather than the residue heavy atoms, use `Calpha=True` and `CG=False`.
- If the structure contains only Cα atoms (e.g., a coarse-grained model), set `Calpha=True` and `CG=True`.
- The `g_threshold=0.6` is the standard cutoff used in the published protocol. Lower values will detect more (but potentially noisier) entanglements.

**Output:** `$OUTDIR/NCLE/1ZMR_GE.csv`

In [2]:
from NCLEdetector.gaussian_entanglement import GaussianEntanglement
import pandas as pd

# ── Inputs ──────────────────────────────────────────────────────────────────
PDB = f"{DATASTORE}/user_input/reference_structures/1zmr_model_clean.pdb"
CHAIN = "A"
ID = "1ZMR"

# ── Initialize ───────────────────────────────────────────────────────────────
ge = GaussianEntanglement(g_threshold=0.6, density=1.0, Calpha=False, CG=False,
                          ent_detection_method=3)

# ── Run ──────────────────────────────────────────────────────────────────────
ge.calculate_native_entanglements(pdb_file=PDB, outdir=NCLE_outdir, ID=ID, chain=CHAIN)


result_step2 = pd.read_csv(f"{NCLE_outdir}/1ZMR_NCLE.csv", sep='|')
print(f"Detected entanglements: {len(result_step2)}")
result_step2[result_step2['ENT']==True].head(100)

2026-10-30 13:22:54 [NCLEdetector.GaussianEntanglement] INFO: 
####################################################################################################
COMPUTING ENTANGLEMENTS FOR 1zmr_model_clean with ID 1ZMR
2026-10-30 13:22:54 [NCLEdetector.GaussianEntanglement] INFO: /home/ians/STORE/NCLEdetector/outputs/notebook_tmp/workflow1/NCLE/1ZMR_NCLE.csv ALREADY EXISTS AND WILL BE LOADED


Detected entanglements: 798


,ID,chain,i,j,crossingsN,crossingsC,gn,gc,GLNn,GLNc,TLNn,TLNc,CCbond,ENT
59,1ZMR,A,15,99,NaN,107.0,-0.03377,1.01395,0,1,0,1,False,True
60,1ZMR,A,15,101,NaN,108.0,-0.02786,0.91393,0,1,0,1,False,True
149,1ZMR,A,39,166,18.0,NaN,0.86554,0.01968,1,0,1,0,False,True
150,1ZMR,A,40,166,18.0,NaN,0.81444,-0.00027,1,0,1,0,False,True
155,1ZMR,A,43,139,17.0,NaN,0.79379,0.30363,1,0,1,0,False,True
156,1ZMR,A,43,165,17.0,NaN,0.83725,-0.03072,1,0,1,0,False,True
157,1ZMR,A,43,166,18.0,NaN,0.81764,-0.01722,1,0,1,0,False,True
163,1ZMR,A,46,165,17.0,NaN,0.76778,-0.02968,1,0,1,0,False,True
173,1ZMR,A,52,102,NaN,108.0,0.26638,0.61363,0,1,0,1,False,True
177,1ZMR,A,53,101,NaN,108.0,0.20966,0.71005,0,1,0,1,False,True


## Step 3. Filter for high-quality entanglements

This step is especially important when using **AlphaFold** or other predicted structures, or when you want to remove slipknots and low-confidence entanglements from the set.

| `model` value | Use when |
|---------------|----------|
| `"EXP"` | Input structure is from X-ray crystallography, cryo-EM, or NMR |
| `"AF"` | Input structure is an AlphaFold prediction |

> *Note:* If the structure contains residues outside the canonical protein sequence (e.g., expression tags, fusion segments), a **mapping file** can be supplied to restrict the filtered set to the canonical region only. See the `select_high_quality_entanglements` documentation for the expected format.

**Output:** `$OUTDIR/HQ_NCLE/1ZMR.csv`

In [3]:
from NCLEdetector.gaussian_entanglement import GaussianEntanglement

PDB = f"{DATASTORE}/user_input/reference_structures/1zmr_model_clean.pdb"
ID = "1ZMR"

# Point to Step 3 output
NCLE_file = f"{NCLE_outdir}/1ZMR_NCLE.csv"

ge = GaussianEntanglement(g_threshold=0.6, density=0.0, Calpha=False, CG=False,
                          ent_detection_method=2)

ge.select_high_quality_entanglements(
    NCLE_file,
    PDB,
    outdir=HQ_NCLE_outdir,
    ID=ID,
    model="EXP",   # "EXP" for experimental structures; "AF" for AlphaFold models
    chain=CHAIN,
)

result_step3 = pd.read_csv(f"{HQ_NCLE_outdir}/1ZMR.csv", sep='|')
print(f"High-quality entanglements: {len(result_step3)}")
result_step3.head()

2026-10-30 13:22:54 [NCLEdetector.GaussianEntanglement] INFO: GE FILE: /home/ians/STORE/NCLEdetector/outputs/notebook_tmp/workflow1/NCLE/1ZMR_NCLE.csv
2026-10-30 13:22:54 [NCLEdetector.GaussianEntanglement] INFO: 
##################################################
Removing slipknots...
2026-10-30 13:22:54 [NCLEdetector.GaussianEntanglement] INFO: SAVED: /home/ians/STORE/NCLEdetector/outputs/notebook_tmp/workflow1/HQ_NCLE/1ZMR.csv


High-quality entanglements: 50


,ID,chain,i,j,crossingsN,crossingsC,gn,gc,GLNn,GLNc,TLNn,TLNc,CCbond,ENT,Slipknot_N,Slipknot_C
0,1ZMR,A,15,99,NaN,107.0,-0.03377,1.01395,0,1,0,1,False,True,False,False
1,1ZMR,A,15,101,NaN,108.0,-0.02786,0.91393,0,1,0,1,False,True,False,False
2,1ZMR,A,39,166,18.0,NaN,0.86554,0.01968,1,0,1,0,False,True,False,False
3,1ZMR,A,40,166,18.0,NaN,0.81444,-0.00027,1,0,1,0,False,True,False,False
4,1ZMR,A,43,139,17.0,NaN,0.79379,0.30363,1,0,1,0,False,True,False,False


## Step 4. Cluster redundant NCLEs into representative entanglements

Many entanglements are structurally degenerate variants of the same topological motif. This step collapses them into a non-redundant representative set.

> *Note:* The `organism` parameter sets the distance threshold optimized on one of three model organisms: `"Ecoli"` for *E. coli*, `"Human"` for *H. sapiens*, and `"Yeast"` for *S. cerevisiae*. Use the `cut_off` parameter to override the organism-specific default.

**Output:** `$OUTDIR/clustered_HQ_NCLE/1ZMR.csv`

In [4]:
from NCLEdetector.clustering import ClusterNativeEntanglements

outfile = "1ZMR.csv"

# Point to Step 4 output
NCLE_file = f"{HQ_NCLE_outdir}/1ZMR.csv"

clustering = ClusterNativeEntanglements(organism="Ecoli", cut_off=None)
clustering.Cluster_NativeEntanglements(
    NCLE_file,    
    ID=ID,
    outdir=clustered_HQ_NCLE_outdir,
    chain=CHAIN,
)
result_step4 = pd.read_csv(f"{clustered_HQ_NCLE_outdir}/1ZMR.csv", sep='|')
print(f"Representative entanglements: {len(result_step4)}")
result_step4.head()

2026-10-30 13:22:54 [NCLEdetector.ClusterNativeEntanglements] INFO: Clustering Ecoli Native Entanglements with dist_cutoff: 57
2026-10-30 13:22:54 [NCLEdetector.ClusterNativeEntanglements] INFO: /home/ians/STORE/NCLEdetector/outputs/notebook_tmp/workflow1/clustered_HQ_NCLE/1ZMR.csv ALREADY EXISTS AND WILL BE LOADED


Representative entanglements: 5


,ID,chain,i,j,crossingsN,crossingsC,gn,gc,GLNn,GLNc,TLNn,TLNc,num_contacts,contacts,CCBond
0,1ZMR,A,230,277,221.0,-283.0,0.71330,-0.16612,1,0,1,-1,16,230-277;228-274;229-277;229-283;230-278;231-27...,False
1,1ZMR,A,55,99,NaN,108.0,0.15095,0.66113,0,1,0,1,7,15-101;52-102;53-101;53-102;54-101;55-99;15-99,False
2,1ZMR,A,43,139,17.0,NaN,0.79379,0.30363,1,0,1,0,6,43-139;43-165;46-165;39-166;40-166;43-166,False
3,1ZMR,A,204,305,187.0,NaN,0.71386,0.24994,1,0,1,0,11,204-305;197-307;200-370;200-371;203-370;203-37...,False
4,1ZMR,A,271,287,-257.0,NaN,-0.86397,0.05277,-1,0,-1,0,10,265-315;267-317;267-318;268-317;268-318;271-28...,False


## Step 5. Compute structural features for representative NCLEs

Once you have the representative set, calculate structural features that characterize each entanglement's complexity, including:

- supercoiling characteristics of the entangled terminus;
- contact patterns between the loop and the rest of the structure;
- other structural descriptors capturing topological complexity.

**Output:** `$OUTDIR/clustered_HQ_NCLE_features/P00558_1ZMR_A_uent_features.csv`

In [5]:
from NCLEdetector.entanglement_features import FeatureGen

PDB = f"{DATASTORE}/user_input/reference_structures/1zmr_model_clean.pdb"
pdb = PDB
UNIPROT = "P00558"
PDBID = "1ZMR"

# Point to Step 5 output
cluster_file = f"{clustered_HQ_NCLE_outdir}/1ZMR.csv"

FGen = FeatureGen()

# gene  : UniProt accession for ecPGK
# chain : PDB chain identifier
# pdbid : four-letter PDB code (uppercase)
EntFeatures = FGen.get_uent_features(pdb_file=PDB, outdir=clustered_HQ_NCLE_feature_outdir, cluster_file=cluster_file, gene=UNIPROT, chain=CHAIN, pdbid=PDBID)
print(EntFeatures)

print(f'Loading feature data from {clustered_HQ_NCLE_feature_outdir}/P00558_1ZMR_A_uent_features.csv')
result_step5 = pd.read_csv(f"{clustered_HQ_NCLE_feature_outdir}/P00558_1ZMR_A_uent_features.csv", sep='|')
print(f"Feature rows: {len(result_step5)}")
result_step5.head()

2026-10-30 13:22:55 [NCLEdetector.FeatureGen] INFO: Unique entanglement features saved to /home/ians/STORE/NCLEdetector/outputs/notebook_tmp/workflow1/clustered_HQ_NCLE_features/P00558_1ZMR_A_uent_features.csv


{'outfile': '/home/ians/STORE/NCLEdetector/outputs/notebook_tmp/workflow1/clustered_HQ_NCLE_features/P00558_1ZMR_A_uent_features.csv', 'results':      gene   PDB chain  ENT-ID       gn  N_term_thread       gc  C_term_thread  \
0  P00558  1ZMR     A       0  0.71330              1 -0.16612              1   
1  P00558  1ZMR     A       1  0.15095              0  0.66113              1   
2  P00558  1ZMR     A       2  0.79379              1  0.30363              0   
3  P00558  1ZMR     A       3  0.71386              1  0.24994              0   
4  P00558  1ZMR     A       4 -0.86397              1  0.05277              0   

     i    j  ... min_N_prot_depth_left min_N_thread_depth_left  \
0  230  277  ...              0.529976                0.960870   
1   55   99  ...                   NaN                     NaN   
2   43  139  ...              0.040767                0.395349   
3  204  305  ...              0.448441                0.916667   
4  271  287  ...              0.61630

,gene,PDB,chain,ENT-ID,gn,N_term_thread,gc,C_term_thread,i,j,...,min_N_prot_depth_left,min_N_thread_depth_left,min_N_thread_slippage_left,min_C_prot_depth_right,min_C_thread_depth_right,min_C_thread_slippage_right,prot_size,ACO,RCO,CCBond
0,P00558,1ZMR,A,0,0.71330,1,-0.16612,1,230,277,...,0.529976,0.960870,221.0,0.321343,0.957143,134.0,417,49.500000,0.118705,False
1,P00558,1ZMR,A,1,0.15095,0,0.66113,1,55,99,...,NaN,NaN,NaN,0.741007,0.971698,309.0,417,58.285714,0.139774,False
2,P00558,1ZMR,A,2,0.79379,1,0.30363,0,43,139,...,0.040767,0.395349,17.0,NaN,NaN,NaN,417,118.833333,0.284972,False
3,P00558,1ZMR,A,3,0.71386,1,0.24994,0,204,305,...,0.448441,0.916667,187.0,NaN,NaN,NaN,417,155.090909,0.371921,False
4,P00558,1ZMR,A,4,-0.86397,1,0.05277,0,271,287,...,0.616307,0.948339,257.0,NaN,NaN,NaN,417,43.100000,0.103357,False


## Step 6. Interactively visualize the representative entanglements

This NGLView widget uses the same interactive NCLE selector as [NCLEexamples](NCLEexamples.ipynb). Choose an NCLE to highlight its loop interior, native-contact endpoints, and crossing residues, or select **Full structure**.

| Colour | Meaning |
|--------|---------|
| Light grey | Rest of chain A |
| **Red** | Loop interior (`i + 1` → `j - 1`) |
| **Orange spheres** | Native-contact endpoints (`i`, `j`) |
| **Blue** + spheres | Crossing residues (`crossingsN` / `crossingsC`) |

Enable Jupyter widgets for the interactive viewer. If needed, install `nglview` and `ipywidgets` in the notebook environment with `python -m pip install nglview ipywidgets`.

In [6]:
import ipywidgets as widgets
import nglview as nv
import pandas as pd
from IPython.display import display

clustered_ncles = pd.read_csv(
    f"{clustered_HQ_NCLE_outdir}/1ZMR.csv",
    sep="|",
    dtype={"crossingsN": str, "crossingsC": str},
).reset_index(drop=True)

ncles = []
for number, row in clustered_ncles.iterrows():
    crossings = []
    crossing_labels = []
    for column, terminus in (("crossingsN", "N"), ("crossingsC", "C")):
        value = row[column]
        if pd.notna(value) and str(value).strip():
            tokens = [token.strip() for token in str(value).split(",")]
            crossings.extend(abs(int(float(token))) for token in tokens if token)
            crossing_labels.append(f"{terminus}-cross: {value}")

    start, end = int(row["i"]), int(row["j"])
    ncles.append({
        "label": f"NCLE {number + 1}: loop {start}-{end} | " + " | ".join(crossing_labels),
        "start": start,
        "end": end,
        "crossings": crossings,
    })

view = nv.show_file(str(PDB), default_representation=False)
view.layout = widgets.Layout(width="100%", height="500px")
selector = widgets.Dropdown(
    options=[("Full structure", -1)] + [(ncle["label"], i) for i, ncle in enumerate(ncles)],
    value=0 if ncles else -1,
    description="NCLE:",
    layout=widgets.Layout(width="100%"),
)
legend = widgets.HTML()

def update_view(change=None):
    legend.value = (
        "Orange VDW spheres: native contacts (i, j) | Red: loop interior | Blue VDW spheres: crossing residues"
        if ncles else "No representative NCLEs retained. Showing the input structure."
    )
    view.clear_representations()
    view.add_cartoon(selection=":A", color="#bfbfbf")
    if selector.value >= 0:
        ncle = ncles[selector.value]
        if ncle["end"] - ncle["start"] > 1:
            loop = f"{ncle['start'] + 1}-{ncle['end'] - 1}:A"
            view.add_cartoon(selection=loop, color="#E57373")
        contacts = f"{ncle['start']}:A or {ncle['end']}:A"
        view.add_spacefill(selection=contacts, color="#FB8C00", radius_type="vdw", radius_scale=1.0)
        if ncle["crossings"]:
            crossings = " or ".join(f"{residue}:A" for residue in ncle["crossings"])
            view.add_cartoon(selection=crossings, color="#1E88E5")
            view.add_spacefill(selection=crossings, color="#1E88E5", radius_type="vdw", radius_scale=1.0)

selector.observe(update_view, names="value")
update_view()
view.center(selection=":A")
display(widgets.VBox([selector, legend, view]))



## I/O Reference for run_nativeNCLE.py

Each file below is listed once, followed by the column-level description table for that file.

### `$DATASTORE/user_input/reference_structures/1zmr_model_clean.pdb`

| I/O | File | File Description |
|---|---|---|
| Input | `$DATASTORE/user_input/reference_structures/1zmr_model_clean.pdb` | Cleaned all-atom structure used as the geometric source for NCLE discovery. |

### `$OUTDIR/NCLE/1ZMR_GE.csv`

| Column Name | Column Description |
|---|---|
| ID | Protein or structure identifier for the entanglement record. |
| chain | Chain identifier in the input structure. |
| i | First residue index of the native contact that defines the loop start. |
| j | Second residue index of the native contact that defines the loop end. |
| crossingsN | Comma-separated N-terminal crossing residues, stored with chirality signs such as +109 or -258. |
| crossingsC | Comma-separated C-terminal crossing residues, stored with chirality signs such as +109 or -258. |
| gn | Raw Gaussian entanglement score for the N-terminal side of the loop. |
| gc | Raw Gaussian entanglement score for the C-terminal side of the loop. |
| GLNn | Rounded Gaussian linking number for the N-terminal side, derived from gn. |
| GLNc | Rounded Gaussian linking number for the C-terminal side, derived from gc. |
| TLNn | Topological linking number for the N-terminal side after Topoly-based crossing assignment and buffer filtering. |
| TLNc | Topological linking number for the C-terminal side after Topoly-based crossing assignment and buffer filtering. |
| CCbond | Boolean flag indicating that the loop contact is a covalent C-C bond, typically a disulfide-linked loop. |
| ENT | Boolean flag showing whether the native contact passed the chosen entanglement detection criterion. |

### `$OUTDIR/HQ_NCLE/1ZMR.csv`

| Column Name | Column Description |
|---|---|
| ID | Protein or structure identifier for the filtered entanglement record. |
| chain | Chain identifier in the input structure. |
| i | First residue index of the loop-defining contact. |
| j | Second residue index of the loop-defining contact. |
| crossingsN | N-terminal crossing residues retained after slipknot filtering. |
| crossingsC | C-terminal crossing residues retained after slipknot filtering. |
| gn | Raw Gaussian entanglement score for the N-terminal side. |
| gc | Raw Gaussian entanglement score for the C-terminal side. |
| GLNn | Rounded Gaussian linking number for the N-terminal side. |
| GLNc | Rounded Gaussian linking number for the C-terminal side. |
| TLNn | Topological linking number for the N-terminal side. |
| TLNc | Topological linking number for the C-terminal side. |
| CCbond | Boolean flag indicating a covalent C-C loop closure. |
| ENT | Boolean flag showing whether the record is still classified as an entanglement after quality filtering. |
| Slipknot_N | Boolean flag indicating that the N-terminal crossing pattern is consistent with a slipknot and should be treated cautiously. |
| Slipknot_C | Boolean flag indicating that the C-terminal crossing pattern is consistent with a slipknot and should be treated cautiously. |

### `$OUTDIR/clustered_HQ_NCLE/1ZMR.csv`

| Column Name | Column Description |
|---|---|
| ID | Protein or structure identifier for the clustered representative. |
| chain | Chain identifier in the input structure. |
| i | First residue index of the representative loop contact. |
| j | Second residue index of the representative loop contact. |
| crossingsN | Representative N-terminal crossing residues for the cluster. |
| crossingsC | Representative C-terminal crossing residues for the cluster. |
| gn | Representative raw Gaussian entanglement score for the N-terminal side. |
| gc | Representative raw Gaussian entanglement score for the C-terminal side. |
| GLNn | Representative rounded Gaussian linking number for the N-terminal side. |
| GLNc | Representative rounded Gaussian linking number for the C-terminal side. |
| TLNn | Representative topological linking number for the N-terminal side. |
| TLNc | Representative topological linking number for the C-terminal side. |
| num_contacts | Number of raw entanglement records merged into the representative cluster. |
| contacts | Semicolon-separated list of the loop residue pairs that were merged into the representative cluster. |
| CCBond | Boolean flag indicating whether the representative loop closure is a covalent C-C bond. |

### `$OUTDIR/clustered_HQ_NCLE_features/P00558_1ZMR_A_uent_features.csv`

| Column Name | Column Description |
|---|---|
| gene | UniProt accession used for protein-level annotation and output naming. |
| PDB | Four-letter PDB identifier for the structure. |
| chain | Chain identifier used for feature extraction. |
| ENT-ID | Row index of the representative entanglement within the clustered input file. |
| gn | Representative raw Gaussian entanglement score for the N-terminal side. |
| N_term_thread | Number of N-terminal crossing residues threading through the loop. |
| gc | Representative raw Gaussian entanglement score for the C-terminal side. |
| C_term_thread | Number of C-terminal crossing residues threading through the loop. |
| i | First residue index of the representative loop contact. |
| j | Second residue index of the representative loop contact. |
| NC | Core loop contact residues, stored as the pair i,j. |
| NC_wbuff | Loop residues expanded by the fixed residue buffer used for neighborhood calculations. |
| NC_region | Residues whose side-chain heavy atoms lie within the neighborhood cutoff of the buffered loop residues. |
| crossingsN | Core N-terminal crossing residues, without the buffer expansion. |
| crossingsC | Core C-terminal crossing residues, without the buffer expansion. |
| crossingsN_wbuff | N-terminal crossing residues expanded by the fixed residue buffer. |
| crossingsC_wbuff | C-terminal crossing residues expanded by the fixed residue buffer. |
| crossingsN_region | Residues whose side-chain heavy atoms lie within the neighborhood cutoff of the buffered N-terminal crossings. |
| crossingsC_region | Residues whose side-chain heavy atoms lie within the neighborhood cutoff of the buffered C-terminal crossings. |
| ent_region | Union of the loop neighborhood, crossing neighborhoods, core loop residues, and core crossing residues. |
| loopsize | Loop length in residues, computed as j minus i. |
| num_zipper_nc | Number of raw entanglement contacts that were merged into the representative cluster. |
| perc_bb_loop | Loop length normalized by the full protein length. |
| num_loop_contacting_res | Number of residues whose side-chain heavy atoms contact the loop neighborhood. |
| num_cross_nearest_neighbors | Total number of residues near the N- and C-terminal crossing neighborhoods. |
| ent_coverage | Fraction of the protein covered by the full entangled region. |
| min_N_prot_depth_left | Fractional distance of the closest N-terminal crossing from the protein N-terminus. |
| min_N_thread_depth_left | Fractional distance of the closest N-terminal crossing from the loop start. |
| min_N_thread_slippage_left | Residue offset of the closest N-terminal crossing from the loop start. |
| min_C_prot_depth_right | Fractional distance of the closest C-terminal crossing from the protein C-terminus. |
| min_C_thread_depth_right | Fractional distance of the closest C-terminal crossing from the loop end. |
| min_C_thread_slippage_right | Residue offset of the closest C-terminal crossing from the protein C-terminus. |
| prot_size | Full protein length from the UniProt canonical sequence. |
| ACO | Average contact order of the merged loop contacts in the representative cluster. |
| RCO | Relative contact order, computed as ACO divided by prot_size. |
| CCBond | Boolean flag indicating whether the representative loop closure is a covalent C-C bond. |
